In [2]:
pip install notebook pandas matplotlib openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Retail Sales Data Analysis
**Company:** Infinite Business Ventures (Pvt) Ltd
**Author:** Tharindu Kothalawala

## Project goal
Clean and analyze a retail sales dataset, create charts, and build an automated Excel report to support business decisions.

## Tools
Python, pandas, matplotlib, openpyxl

## AI disclosure
I used Claude (AI assistant) for guidance on the approach. I can explain and modify every part of this solution.

## Step 1: Load and inspect the raw data
The raw Excel sheet is never edited. I load it into Python with pandas and work on a copy (`df`). The original (`raw`) stays unchanged.


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl

# Load the raw data (this does NOT change the Excel file)
FILE = "Retail_Sales_Intern_Dataset.xlsx"
raw = pd.read_excel(FILE, sheet_name="Raw_Sales_Data")

# Keep raw safe. We will only work on a copy.
df = raw.copy()

# Inspect the data
print("Rows, columns:", raw.shape)
print("\nData types:\n", raw.dtypes)
print("\nMissing values:\n", raw.isna().sum())
print("\nExact duplicate rows:", raw.duplicated().sum())
print("Duplicate invoice numbers:", raw["Invoice_No"].duplicated().sum())

# Look at the text columns
print("\nBrands:\n", raw["Brand"].value_counts(dropna=False))
print("\nOutlets:\n", raw["Outlet"].value_counts(dropna=False))

# Look at the number columns
print(raw[["Quantity", "Unit_Price_LKR", "Discount_Rate", "Net_Sales_LKR"]].describe())

Rows, columns: (240, 9)

Data types:
 Invoice_No                object
Date              datetime64[ns]
Brand                     object
Product                   object
Outlet                    object
Quantity                 float64
Unit_Price_LKR             int64
Discount_Rate            float64
Net_Sales_LKR            float64
dtype: object

Missing values:
 Invoice_No        0
Date              1
Brand             1
Product           0
Outlet            0
Quantity          1
Unit_Price_LKR    0
Discount_Rate     0
Net_Sales_LKR     1
dtype: int64

Exact duplicate rows: 1
Duplicate invoice numbers: 1

Brands:
 Brand
Protecto         33
Raja & Rani      32
Safety Sure      31
Be Dapper        29
Giggles          29
Be Desi          28
Sultan           28
Lux Desi         27
NaN               1
Unknown Brand     1
 be desi          1
Name: count, dtype: int64

Outlets:
 Outlet
Wellawatte        78
Online            75
Dehiwala          69
Head Office       17
Unknown Outlet     1
N

### What I found
- The dataset has 240 rows and 9 columns.
- Missing values: Date (1), Brand (1), Quantity (1), Net_Sales_LKR (1).
- There is 1 exact duplicate row (INV-10018).
- Brand has inconsistent text (" be desi") and an "Unknown Brand" value.
- Outlet has an "Unknown Outlet" value.
- Some numbers break the rules: a negative quantity, a negative price, and a discount above 100%.
- 

## Step 2: Clean the data
I clean a copy of the data (`df`). The raw data (`raw`) is never changed.
A `DQ_Flag` column records every problem found in each row, so nothing is changed silently.

### 2.1 Duplicate rows
**Problem:** Invoice INV-10018 appears twice with exactly the same values.
**Decision:** Remove the second copy and keep one.
**Reason:** Identical rows on the same invoice are a double entry. Keeping both would count the sale twice and inflate net sales.

In [5]:
# Start fresh from the raw data
df = raw.copy()

# Column to record data-quality notes for each row
df["DQ_Flag"] = ""

# Helper function: adds a note to the rows selected by 'mask'
def add_flag(mask, message):
    df.loc[mask, "DQ_Flag"] = df.loc[mask, "DQ_Flag"] + message + "; "

# --- 2.1 Duplicates ---
dup_mask = df.duplicated(keep="first")
print("Duplicate rows found:", dup_mask.sum())

# Show the duplicated invoice so we can see it
print(df[df["Invoice_No"].isin(df.loc[dup_mask, "Invoice_No"])])

rows_before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print("Rows before:", rows_before, "| Rows after:", len(df))

Duplicate rows found: 1
   Invoice_No       Date    Brand Product    Outlet  Quantity  Unit_Price_LKR  \
17  INV-10018 2026-07-12  Giggles    Doll  Dehiwala       1.0           22500   
18  INV-10018 2026-07-12  Giggles    Doll  Dehiwala       1.0           22500   

    Discount_Rate  Net_Sales_LKR DQ_Flag  
17            0.0        22500.0          
18            0.0        22500.0          
Rows before: 240 | Rows after: 239


### 2.2 Brand column
**Problem 1 (inconsistent text):** One brand is written as " be desi" (lowercase, extra space).
**Decision:** Remove extra spaces and match it to the correct brand name (`Be Desi`).
**Reason:** Otherwise pandas treats it as a different brand and splits Be Desi's sales into two groups.

**Problem 2 (missing / unknown brand):** One brand is blank and one says "Unknown Brand".
**Decision:** Label both as `Unknown Brand` and flag them. I keep the rows.
**Reason:** I cannot guess the brand safely (the same product, like Gloves, exists in more than one brand). The quantity, price and net sales are still valid, so removing the rows would understate total sales.

In [6]:
# --- 2.2 Brand ---
# The 8 correct brand names (from the dataset)
known_brands = ["Be Dapper", "Be Desi", "Giggles", "Lux Desi",
                "Protecto", "Raja & Rani", "Safety Sure", "Sultan"]

# Lookup: lowercase name -> correct name
brand_lookup = {b.lower(): b for b in known_brands}

original_brand = df["Brand"].copy()

# Remove extra spaces, then match to the correct spelling
cleaned_brand = df["Brand"].str.strip().str.lower().map(brand_lookup)

# Rows where the text was fixed (it had a value, and the value changed)
text_fixed = cleaned_brand.notna() & (original_brand != cleaned_brand)
add_flag(text_fixed, "Brand text standardised")

# Rows with a blank brand
missing_brand = original_brand.isna()
add_flag(missing_brand, "Brand missing")

# Rows that already say "Unknown Brand"
unknown_brand = original_brand == "Unknown Brand"
add_flag(unknown_brand, "Brand unknown")

# Apply the cleaned brand; everything that did not match becomes "Unknown Brand"
df["Brand"] = cleaned_brand.fillna("Unknown Brand")

print("Text fixed:", text_fixed.sum())
print("Missing brand:", missing_brand.sum())
print("Already unknown:", unknown_brand.sum())
print()
print(df["Brand"].value_counts())

Text fixed: 1
Missing brand: 1
Already unknown: 1

Brand
Protecto         33
Raja & Rani      32
Safety Sure      31
Be Desi          29
Be Dapper        29
Sultan           28
Giggles          28
Lux Desi         27
Unknown Brand     2
Name: count, dtype: int64


### 2.3 Outlet and Date
**Problem 1 (unknown outlet):** One row has "Unknown Outlet". The Data Dictionary allows only Dehiwala, Wellawatte, Head Office and Online.
**Decision:** Keep the row, label it `Unknown Outlet`, and flag it.
**Reason:** The sale amount is valid. I cannot guess which outlet it belongs to, and deleting it would lower total sales.

**Problem 2 (missing date):** One row (INV-10078) has no date.
**Decision:** Keep the row and flag it. I do not invent a date.
**Reason:** The sale value is valid, so it counts in total sales. But it cannot be placed in a month, so the Monthly Summary will be slightly lower than total sales. I will state this clearly in the report.

In [7]:
# --- 2.3 Outlet ---
valid_outlets = ["Dehiwala", "Wellawatte", "Head Office", "Online"]

# Remove extra spaces first
df["Outlet"] = df["Outlet"].str.strip()

# Anything not in the valid list is unknown
bad_outlet = ~df["Outlet"].isin(valid_outlets)
add_flag(bad_outlet, "Outlet unknown")
df.loc[bad_outlet, "Outlet"] = "Unknown Outlet"

print("Unknown outlets:", bad_outlet.sum())
print(df["Outlet"].value_counts())

# --- 2.3 Date ---
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

missing_date = df["Date"].isna()
add_flag(missing_date, "Date missing")

# Month column for the monthly summary (blank when the date is missing)
df["Month"] = df["Date"].dt.strftime("%Y-%m")

print("\nMissing dates:", missing_date.sum())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

Unknown outlets: 1
Outlet
Wellawatte        78
Online            75
Dehiwala          68
Head Office       17
Unknown Outlet     1
Name: count, dtype: int64

Missing dates: 1
Date range: 2026-07-01 to 2026-09-30


### 2.4 Invalid Quantity, Unit Price and Discount
**Problem:** Four rows break the Data Dictionary rules: a blank quantity (INV-10055), a negative quantity (INV-10093), a negative price (INV-10109) and a discount of 125% (INV-10132).
**Decision:** Keep the rows in the cleaned dataset, flag them, and set `Include_In_Analysis = False`. They are excluded from KPIs and summaries.
**Reason:** I cannot know the true value. Net_Sales_LKR hints at a likely value, but using it would be an assumption. These rows are under 1% of total sales, so excluding them has a very small effect.

In [8]:
# --- 2.4 Invalid quantity, price, discount ---
# Start by assuming every row is OK to use
df["Include_In_Analysis"] = True

# Rule checks (from the Data Dictionary)
bad_qty = df["Quantity"].isna() | (df["Quantity"] <= 0) | (df["Quantity"] % 1 != 0)
bad_price = df["Unit_Price_LKR"].isna() | (df["Unit_Price_LKR"] <= 0)
bad_disc = df["Discount_Rate"].isna() | (df["Discount_Rate"] < 0) | (df["Discount_Rate"] > 1)

# Flag each problem
add_flag(bad_qty, "Invalid quantity")
add_flag(bad_price, "Invalid unit price")
add_flag(bad_disc, "Invalid discount")

# Exclude these rows from analysis (they stay in the data)
df.loc[bad_qty | bad_price | bad_disc, "Include_In_Analysis"] = False

print("Invalid quantity:", bad_qty.sum())
print("Invalid price:", bad_price.sum())
print("Invalid discount:", bad_disc.sum())
print("Rows excluded:", (~df["Include_In_Analysis"]).sum())

# Show the excluded rows
print(df.loc[~df["Include_In_Analysis"],
             ["Invoice_No", "Quantity", "Unit_Price_LKR", "Discount_Rate", "Net_Sales_LKR", "DQ_Flag"]])

Invalid quantity: 2
Invalid price: 1
Invalid discount: 1
Rows excluded: 4
    Invoice_No  Quantity  Unit_Price_LKR  Discount_Rate  Net_Sales_LKR  \
53   INV-10055       NaN            1500           0.30         3150.0   
91   INV-10093      -2.0            9990           0.15        25474.5   
107  INV-10109       5.0           -1500           0.00         7500.0   
130  INV-10132       3.0            1500           1.25         4050.0   

                  DQ_Flag  
53     Invalid quantity;   
91     Invalid quantity;   
107  Invalid unit price;   
130    Invalid discount;   


### 2.5 Net Sales validation
**Check:** I calculate `Net_Sales_Calc = Quantity × Unit_Price_LKR × (1 − Discount_Rate)` and compare it with the given `Net_Sales_LKR`. A difference of more than 1 LKR is treated as a mismatch (small tolerance for rounding).
**Decision 1:** Mismatches are flagged, not changed. The original Net_Sales_LKR value is kept.
**Decision 2:** One row (INV-10156) has a blank Net_Sales_LKR but valid quantity, price and discount. I fill it using the formula and flag it.
**Reason:** The formula gives an exact result from valid inputs, so this is a calculation, not a guess.

In [9]:
# --- 2.5 Net sales validation ---
# Our own calculation from the three inputs
df["Net_Sales_Calc"] = df["Quantity"] * df["Unit_Price_LKR"] * (1 - df["Discount_Rate"])

# Difference between given value and our calculation
df["Net_Sales_Diff"] = df["Net_Sales_LKR"] - df["Net_Sales_Calc"]

TOLERANCE = 1  # LKR, allows for small rounding

# Mismatch: both values exist, but they differ by more than the tolerance
mismatch = df["Net_Sales_LKR"].notna() & df["Net_Sales_Calc"].notna() & (df["Net_Sales_Diff"].abs() > TOLERANCE)
add_flag(mismatch, "Net sales mismatch")

# Missing net sales
missing_net = df["Net_Sales_LKR"].isna()
add_flag(missing_net, "Net sales missing")

# Fill only when the inputs are valid (row is included in analysis)
fill_mask = missing_net & df["Include_In_Analysis"]
df.loc[fill_mask, "Net_Sales_LKR"] = df.loc[fill_mask, "Net_Sales_Calc"]
add_flag(fill_mask, "Net sales recalculated")

print("Mismatches:", mismatch.sum())
print("Missing net sales:", missing_net.sum(), "| Recalculated:", fill_mask.sum())

print("\nMismatch rows:")
print(df.loc[mismatch, ["Invoice_No", "Net_Sales_LKR", "Net_Sales_Calc", "Net_Sales_Diff", "Include_In_Analysis"]])

print("\nRecalculated row:")
print(df.loc[fill_mask, ["Invoice_No", "Quantity", "Unit_Price_LKR", "Discount_Rate", "Net_Sales_LKR"]])

Mismatches: 3
Missing net sales: 1 | Recalculated: 1

Mismatch rows:
    Invoice_No  Net_Sales_LKR  Net_Sales_Calc  Net_Sales_Diff  \
91   INV-10093        25474.5        -16983.0         42457.5   
107  INV-10109         7500.0         -7500.0         15000.0   
130  INV-10132         4050.0         -1125.0          5175.0   

     Include_In_Analysis  
91                 False  
107                False  
130                False  

Recalculated row:
    Invoice_No  Quantity  Unit_Price_LKR  Discount_Rate  Net_Sales_LKR
154  INV-10156       3.0           15990            0.0        47970.0


### 2.6 Final checks, save cleaned data, and data-quality summary
I check the cleaned data against the rules, then save it to `outputs/Cleaned_Sales_Data.csv`. The raw Excel file is never changed.

**Summary of cleaning actions**
- Removed: 1 duplicate row (INV-10018)
- Corrected: 1 brand text standardised, 1 net sales value recalculated (INV-10156)
- Excluded from analysis: 4 rows with invalid quantity, price or discount
- Flagged but kept: unknown brand, unknown outlet, missing date, net sales mismatch

**Note:** The row with the missing date (INV-10078) is included in total sales but cannot be placed in a month. The Monthly Summary will be slightly lower than total net sales because of this.

In [10]:
import os

# --- 2.6 Final checks ---
print("Rows in cleaned data:", len(df))
print("Rows included in analysis:", df["Include_In_Analysis"].sum())
print("Rows excluded:", (~df["Include_In_Analysis"]).sum())

# Check the included rows follow the Data Dictionary rules
ok = df[df["Include_In_Analysis"]]
print("\nChecks on included rows (all should be True):")
print("Quantity > 0:", (ok["Quantity"] > 0).all())
print("Unit price > 0:", (ok["Unit_Price_LKR"] > 0).all())
print("Discount between 0 and 1:", ok["Discount_Rate"].between(0, 1).all())
print("No missing net sales:", ok["Net_Sales_LKR"].notna().all())
print("Invoice numbers unique:", ok["Invoice_No"].is_unique)

# --- Save cleaned data separately ---
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/Cleaned_Sales_Data.csv", index=False)
print("\nSaved: outputs/Cleaned_Sales_Data.csv")

# --- Data-quality summary table ---
dq_summary = pd.DataFrame({
    "Action": ["Removed", "Corrected", "Corrected", "Excluded", "Flagged", "Flagged", "Flagged", "Flagged"],
    "Issue": ["Duplicate row", "Brand text standardised", "Net sales recalculated",
              "Invalid quantity/price/discount", "Unknown brand", "Unknown outlet",
              "Missing date", "Net sales mismatch"],
    "Rows": [
        len(raw) - rows_before + (rows_before - len(raw[~raw.duplicated()])),
        text_fixed.sum(),
        fill_mask.sum(),
        (~df["Include_In_Analysis"]).sum(),
        (missing_brand | unknown_brand).sum(),
        bad_outlet.sum(),
        missing_date.sum(),
        mismatch.sum(),
    ],
})
print()
print(dq_summary)

Rows in cleaned data: 239
Rows included in analysis: 235
Rows excluded: 4

Checks on included rows (all should be True):
Quantity > 0: True
Unit price > 0: True
Discount between 0 and 1: True
No missing net sales: True
Invoice numbers unique: True

Saved: outputs/Cleaned_Sales_Data.csv

      Action                            Issue  Rows
0    Removed                    Duplicate row     1
1  Corrected          Brand text standardised     1
2  Corrected           Net sales recalculated     1
3   Excluded  Invalid quantity/price/discount     4
4    Flagged                    Unknown brand     2
5    Flagged                   Unknown outlet     1
6    Flagged                     Missing date     1
7    Flagged               Net sales mismatch     3
